# Agents in LangChain
### Converted from OpenAI → Google Gemini
**ReAct Agent using `gemini-2.5-flash` + DuckDuckGo Search + Weather API**

---

## Step 1 — Install Dependencies

In [18]:
!pip install -q langchain-google-genai langchain-community langchain-core langchain requests duckduckgo-search


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — API Keys

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()


True

## Step 3 — Imports

In [20]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install duckduckgo-search


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
pip install langchainhub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor    
import langchainhub
import requests


## Step 4 — Define Tools

Same two tools as the original:
- `DuckDuckGoSearchRun` — searches the web
- `get_weather_data` — fetches live weather from Weatherstack API

In [36]:
# Built-in search tool
search_tool = DuckDuckGoSearchRun()

print(f'Search tool ready: {search_tool.name}')

Search tool ready: duckduckgo_search


In [37]:
# Custom weather tool
@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f'https://api.weatherstack.com/current?access_key=91f03ed6b0a148448b3ed9f8e4266255&query={city}'
    response = requests.get(url)
    return str(response.json())

print(f'Weather tool ready: {get_weather_data.name}')

Weather tool ready: get_weather_data


## Step 5 — Initialise Gemini Model

Replacing `ChatOpenAI()` with `ChatGoogleGenerativeAI` using `gemini-2.5-flash`.

In [38]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GEMINI_API_KEY']
)

print('Model ready: gemini-2.5-flash')

Model ready: gemini-2.5-flash


## Step 6 — Pull the ReAct Prompt from LangChain Hub

`hwchase17/react` is the standard ReAct prompt used for all ReAct agents.
This is the same as the original — no change needed here.

In [43]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

print('ReAct prompt created successfully')
print(prompt.input_variables)

ReAct prompt created successfully
['agent_scratchpad', 'input', 'tool_names', 'tools']


## Step 7 — Create the ReAct Agent

Same `create_react_agent()` call as original, just using Gemini instead of OpenAI.

In [44]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

print('ReAct agent created')

ReAct agent created


## Step 8 — Wrap with AgentExecutor

`AgentExecutor` runs the agent loop automatically.
`verbose=True` prints every thought, action, and observation — exactly like the original output you saw.

In [45]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True
)

print('AgentExecutor ready')

AgentExecutor ready


## Step 9 — Run the Agent!

Same query as original — find the capital of Madhya Pradesh, then get its weather.

In [46]:
response = agent_executor.invoke({
    'input': 'Find the capital of Madhya Pradesh, then find its current weather condition'
})

print(response['output'])



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of Madhya PradeshIts capital is Bhopal, and Madhya Pradesh is the second-largest Indian state by area and the fifth-largest state by population with over 72 million residents. It borders the states of Uttar Pradesh, Chhattisgarh, Maharashtra, Gujarat, and Rajasthan. ° to 86°). Madhya Pradesh receives an average annual rainfall of about 1200 mm (nearly 50 inches), of which 90 percent falls during the monsoon season. The capital of the state is Bhopal. They unite at the centre of the city where a small 18th century temple of Sangamnath or Indreshwar exists. The name Indore is due to this diety. It is the largest city in Madhya Pradesh state in central India .It is the commercial capital of the state of Madhya Pradesh. Its capital is Bhopal, and the largest city is Indore, with Jabalpur, Gwalior, Ujjain, Satna, Sagar, Rewa and Damoh being the other major cities. Madhya Pradesh is the second largest Ind

## Step 10 — Try More Queries

The agent can chain any number of tool calls. Try these:

In [47]:
# Try 2 — different city
response2 = agent_executor.invoke({
    'input': 'What is the capital of Tamil Nadu and what is the weather there right now?'
})
print(response2['output'])



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of Tamil Nadu4 days ago - Tamil Nadu is the southernmost state of India. The tenth largest Indian state by area and the sixth largest by population, it is the home of the Tamils, who speak the Tamil language, the state's official language and the first to be recognised as a classical language in India. Chennai is the state's largest city and capital... 2 days ago - Chennai, also known as Madras, is the capital and largest city of Tamil Nadu, the southernmost state of India. It is located on the Coromandel Coast of the Bay of Bengal. As per the 2011 Indian census, Chennai is the sixth-most-populous city in India and forms the fourth-most-populous urban ... 2 weeks ago - Millions of people starved to death throughout British ruled Tamil Nadu, around 3.9 million people perished in Chennai alone within two years of 1877–78. Madras was the capital of the Madras Presidency and thus became home to importan

In [49]:
# Try 3 — search + weather combined
response3 = agent_executor.invoke({
    'input': 'Which city hosted IPL 2024 finals and what is the current weather in that city?'
})
print(response3['output'])



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: Which city hosted IPL 2024 finalsThe 2024 Indian Premier League (also known as IPL 17 and branded as TATA IPL 2024) was the 17th edition of the Indian Premier League. The tournament featured ten teams competing in 74 matches from 22 March to 26 May 2024. It was held across 13 cities in India, with Chennai hosting the opening ceremony and the final as the defending champions. Chennai will host the IPL 2024 final for the first time in the league's 12-year history. The M. A. Chidambaram Stadium, home of the Chennai Super Kings and last year's IPL winners has been named ... Notably, since its inception in 2020, Narendra Modi Stadium, with a total seating capacity of 132,000, has already hosted three IPL finals, and adding the 2026 edition would take the tally to four. That being said, explore the complete list of IPL Final venues over the years from 2008 to 2026. IPL Final venues (2008-2026) After a long 12-yea

---
## What Changed from the Original

| Original (OpenAI) | Converted (Gemini) |
|---|---|
| `from langchain_openai import ChatOpenAI` | `from langchain_google_genai import ChatGoogleGenerativeAI` |
| `os.environ['OPENAI_API_KEY'] = ...` | `os.environ['GEMINI_API_KEY'] = ...` |
| `llm = ChatOpenAI()` | `llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', google_api_key=...)` |
| `!pip install langchain-openai` | `!pip install langchain-google-genai` |

Everything else — `create_react_agent`, `AgentExecutor`, `hub.pull`, `@tool`, `DuckDuckGoSearchRun` — stays **exactly the same**.

---
## How the ReAct Agent Works

ReAct stands for **Reason + Act**. The agent follows this loop internally:

```
Thought: I need to find the capital of Madhya Pradesh first
Action: duckduckgo_search
Action Input: capital of Madhya Pradesh
Observation: Bhopal is the capital...
Thought: Now I need the weather for Bhopal
Action: get_weather_data
Action Input: Bhopal
Observation: {temperature: 40, ...}
Thought: I have everything I need
Final Answer: The capital is Bhopal, currently 40°C and partly cloudy.
```

The `verbose=True` in `AgentExecutor` prints this entire reasoning process so you can see exactly how the agent thinks step by step. This is the same loop you built manually in the tool calling lecture — `AgentExecutor` just automates it.